# Анализ экспериментов TGNv2

Рабочий ноутбук для анализа прогонов из локального MLflow (SQLite).

**Соглашения проекта:** визуализация — Plotly, датафреймы — Polars, текст и комментарии — на русском.

In [1]:
import polars as pl
import plotly.express as px
import mlflow

# Тот же локальный бэкенд, что и в train-tgbn-nodeproppred.py
mlflow.set_tracking_uri("sqlite:///mlruns/mlflow.db")
print("polars", pl.__version__, "| plotly", px.__version__ if hasattr(px, "__version__") else "?", "| mlflow", mlflow.__version__)

polars 1.41.2 | plotly ? | mlflow 3.11.1


## Загрузка прогонов из MLflow

Тянем все раны во всех экспериментах в один Polars-датафрейм.

In [2]:
exp_ids = [e.experiment_id for e in mlflow.search_experiments()]
runs = mlflow.search_runs(experiment_ids=exp_ids, output_format="list") if exp_ids else []

rows = [
    {
        "run": r.info.run_name,
        "model": r.data.tags.get("model"),
        "dataset": r.data.tags.get("dataset"),
        "seed": r.data.tags.get("seed"),
        "best_val_ndcg": r.data.metrics.get("best_val_ndcg"),
        "best_test_ndcg": r.data.metrics.get("best_test_ndcg"),
        "best_epoch": r.data.metrics.get("best_epoch"),
    }
    for r in runs
]
df = pl.DataFrame(rows)
print(f"Найдено прогонов: {df.height}")
df

2026/06/23 16:47:19 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/06/23 16:47:19 INFO mlflow.store.db.utils: Updating database tables


Найдено прогонов: 0


shape: (0, 0)
┌┐
╞╡
└┘

## Сравнение test NDCG по моделям и датасетам

In [3]:
if df.height and df["best_test_ndcg"].null_count() < df.height:
    fig = px.bar(
        df.drop_nulls("best_test_ndcg").to_pandas(),
        x="dataset", y="best_test_ndcg", color="model", barmode="group",
        title="Лучший test NDCG по датасетам и моделям",
    )
    fig.show()
else:
    print("Пока нет завершённых прогонов с метрикой best_test_ndcg — запусти обучение.")

Пока нет завершённых прогонов с метрикой best_test_ndcg — запусти обучение.
